# Step 2 — Feature Engineering EDA

**Purpose**: Exploratory analysis of the engineered feature space.  
All plots are **descriptive only** — they inform the paper but do **not** drive feature selection (that lives inside CV in Step 3).

Sections:
- 2.2 Feature family redundancy
- 2.3 Shape feature cross-sequence consistency
- 2.4 Delta feature distributions by class
- 2.5 Temporal feature distributions by class
- 2.6 Sequence length distribution
- 2.7 UMAP visualization
- 2.8 Temporal autocorrelation (t vs t+1)

⚠️ Sections 2.4, 2.5, 2.7 use the `target` label for colouring — descriptive only.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import umap
from plotly.subplots import make_subplots

# Paths
PARQUET = Path("../data/processed/dataset_engineered.parquet")
FIGURES = Path("figures")
FIGURES.mkdir(parents=True, exist_ok=True)

# Seaborn theme for paper figures
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
PAPER_FIG_DPI = 150

# Class palette (consistent across all plots)
CLASS_PALETTE = {"Progressive": "#e74c3c", "Stable": "#3498db", "Response": "#2ecc71"}
CLASS_ORDER = ["Progressive", "Stable", "Response"]

df = pd.read_parquet(PARQUET)
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Patients: {df['Patient'].nunique()}")
print(f"Class distribution:\n{df['target'].value_counts()}")

In [ ]:
# Shape
print(f"Shape: {df.shape}")
print()

# Dtypes summary
print("Dtype counts:")
print(df.dtypes.value_counts())
print()

# Column groups by prefix
import re
from collections import defaultdict

groups = defaultdict(list)
for col in df.columns:
    if col in ["Patient", "Timepoint", "week_num", "target", "target_encoded"]:
        groups["id_target"].append(col)
    elif col.startswith("delta_CE_NC_ratio") or col.startswith("delta_CE_vs_nadir"):
        groups["delta_derived"].append(col)
    elif col in ["CE_NC_ratio","ED_CE_ratio","CE_fraction","total_tumor_volume",
                 "CE_vs_nadir","weeks_since_nadir","is_nadir_scan"]:
        groups["derived"].append(col)
    elif col in ["interval_weeks","scan_index","time_from_diagnosis_weeks"]:
        groups["temporal_meta"].append(col)
    elif col.startswith("delta_"):
        groups["delta_radiomic"].append(col)
    else:
        groups["radiomic"].append(col)

for g, cols in groups.items():
    print(f"{g}: {len(cols)} columns")

print()
# Sample of radiomic naming convention
print("Radiomic sample (first 5):")
print(groups["radiomic"][:5])
print()
print("Delta radiomic sample (first 5):")
print(groups["delta_radiomic"][:5])

print()
# NaN check
print("NaN per group:")
for g, cols in groups.items():
    nan_count = df[cols].isna().sum().sum()
    print(f"  {g}: {nan_count} NaN total")

print()
# Class distribution
print("Class distribution (target):")
print(df["target"].value_counts())

print([c for c in df.columns if 'baseline' in c.lower()])
# atteso: ['is_baseline_scan']
print(df['is_baseline_scan'].value_counts())
# atteso: ~68 True (uno per paziente), ~163 False

# Verifica esatta del gruppo radiomic — esclude is_baseline_scan
radiomic_exact = [c for c in df.columns 
                  if c not in ["Patient", "Timepoint", "target", "target_encoded",
                               "is_baseline_scan", "is_nadir_scan",
                               "CE_NC_ratio", "ED_CE_ratio", "CE_fraction", 
                               "total_tumor_volume", "CE_vs_nadir", 
                               "weeks_since_nadir",
                               "interval_weeks", "scan_index", 
                               "time_from_diagnosis_weeks"]
                  and not c.startswith("delta_")]

delta_radiomic_exact = [c for c in df.columns 
                        if c.startswith("delta_") 
                        and c not in ["delta_CE_NC_ratio", "delta_CE_vs_nadir"]]

print(f"Radiomic assoluti (esatti): {len(radiomic_exact)}")
print(f"Delta radiomic (esatti):    {len(delta_radiomic_exact)}")
print(f"Radiomic == Delta radiomic: {len(radiomic_exact) == len(delta_radiomic_exact)}")

# Verifica simmetria: ogni radiomic ha il suo delta?
missing_deltas = [c for c in radiomic_exact if f"delta_{c}" not in df.columns]
orphan_deltas  = [c for c in delta_radiomic_exact if c.replace("delta_", "", 1) not in radiomic_exact]
print(f"Radiomic senza delta: {len(missing_deltas)}")
print(f"Delta orfani:         {len(orphan_deltas)}")

# Verifica is_baseline_scan non sia nel gruppo radiomic
print(f"\nis_baseline_scan in radiomic_exact: {'is_baseline_scan' in radiomic_exact}")

# Verifica dtype dei bool flags
print(f"\ndtype is_baseline_scan: {df['is_baseline_scan'].dtype}")
print(f"dtype is_nadir_scan:    {df['is_nadir_scan'].dtype}")

# Conteggio finale per gruppo
print(f"\n--- Breakdown finale ---")
print(f"radiomic:       {len(radiomic_exact)}")
print(f"delta_radiomic: {len(delta_radiomic_exact)}")
print(f"derived:        4 (cross-compartment) + 2 (nadir continuous) = 6")
print(f"delta_derived:  2")
print(f"temporal_meta:  3")
print(f"binary_flags:   2")
print(f"id_target:      4")
total = len(radiomic_exact) + len(delta_radiomic_exact) + 6 + 2 + 3 + 2 + 4
print(f"TOTALE:         {total}  (atteso 2585)")

In [ ]:
# ---------------------------------------------------------------------------
# Column catalogue — used across sections
# ---------------------------------------------------------------------------

DERIVED_FEATURES = [
    "CE_NC_ratio", "ED_CE_ratio", "CE_fraction", "total_tumor_volume",
    "CE_vs_nadir", "weeks_since_nadir", "is_nadir_scan",
    "delta_CE_NC_ratio", "delta_CE_vs_nadir",
]

TEMPORAL_FEATURES = [
    "interval_weeks", "scan_index", "time_from_diagnosis_weeks", "weeks_since_nadir",
]

# PyRadiomics families
FAMILIES = ["shape", "firstorder", "glcm", "glrlm", "glszm", "gldm", "ngtdm"]

def get_family_cols(df: pd.DataFrame, family: str) -> list[str]:
    return [c for c in df.columns if f"original_{family}_" in c]

# Radiomic feature columns (original, no _prev suffix, no derived)
radiomic_cols = [
    c for c in df.columns
    if any(f"original_{f}_" in c for f in FAMILIES)
    and not c.endswith("_prev")
    and not c.startswith("delta_")  # ← aggiungere
]
print(f"Radiomic features: {len(radiomic_cols)}")


print(df.shape)
print(df.dtypes.value_counts())
# breakdown colonne per prefisso/gruppo

---
## 2.2 — Feature family redundancy

Inter-feature Pearson correlation within each compartment block, grouped by PyRadiomics family.  
Expected: GLRLM, GLSZM, GLDM highly correlated (>0.9). Shape and firstorder less so.

In [ ]:
print([c for c in df.columns if c.startswith("delta_")][:3])

In [ ]:
COMPARTMENTS = ["CE", "NC", "ED"]

for compartment in COMPARTMENTS:
    # Columns for this compartment, no _prev
    comp_cols = [
        c for c in radiomic_cols
        if c.startswith(f"{compartment}_")
    ]
    if not comp_cols:
        print(f"No columns found for {compartment}")
        continue

    corr = df[comp_cols].corr().abs()

    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(
        corr, ax=ax, cmap="rocket_r", vmin=0, vmax=1,
        xticklabels=False, yticklabels=False,
        cbar_kws={"label": "|Pearson r|"},
    )
    ax.set_title(f"Feature correlation — {compartment} compartment", fontsize=13)
    fig.tight_layout()
    fig.savefig(FIGURES / f"2.2_corr_heatmap_{compartment}.png", dpi=PAPER_FIG_DPI)
    plt.show()

# Family-level redundancy summary
records = []
for compartment in COMPARTMENTS:
    for family in FAMILIES:
        cols = [
            c for c in radiomic_cols
            if c.startswith(f"{compartment}_") and f"original_{family}_" in c
        ]
        if len(cols) < 2:
            continue
        corr_vals = df[cols].corr().abs().values
        upper = corr_vals[np.triu_indices_from(corr_vals, k=1)]
        records.append({
            "compartment": compartment,
            "family": family,
            "n_features": len(cols),
            "mean_|r|": round(upper.mean(), 3),
            "pct_above_0.9": round((upper > 0.9).mean() * 100, 1),
        })

redundancy_df = pd.DataFrame(records)
print(redundancy_df.to_string(index=False))
redundancy_df.to_csv(FIGURES / "2.2_family_redundancy_summary.csv", index=False)

---
## 2.3 — Shape feature cross-sequence consistency

Shape features depend only on the mask, not image intensity → should be identical across sequences (CT1/T1/T2/FLAIR) for the same compartment.  
Differences indicate segmentation inconsistency.

In [ ]:
SEQUENCES = ["CT1", "T1", "T2", "FLAIR"]
SHAPE_FEATS = ["MeshVolume", "Sphericity", "Maximum3DDiameter"]

consistency_records = []

for compartment in COMPARTMENTS:
    for feat in SHAPE_FEATS:
        cols = {
            seq: f"{compartment}_{seq}_original_shape_{feat}"
            for seq in SEQUENCES
        }
        available = {seq: col for seq, col in cols.items() if col in df.columns}
        if len(available) < 2:
            continue

        ref_seq, ref_col = next(iter(available.items()))
        for seq, col in list(available.items())[1:]:
            diff = (df[ref_col] - df[col]).abs()
            consistency_records.append({
                "compartment": compartment,
                "feature": feat,
                "seq_a": ref_seq,
                "seq_b": seq,
                "max_abs_diff": round(diff.max(), 4),
                "mean_abs_diff": round(diff.mean(), 4),
                "n_nonzero": int((diff > 1e-6).sum()),
            })

consistency_df = pd.DataFrame(consistency_records)
print(consistency_df.to_string(index=False))
consistency_df.to_csv(FIGURES / "2.3_shape_consistency.csv", index=False)

# Flag any inconsistencies
flagged = consistency_df[consistency_df["mean_abs_diff"] > 1e-3]
if len(flagged):
    print(f"\n⚠️  {len(flagged)} shape feature pairs show cross-sequence inconsistency — check segmentation.")
else:
    print("\n✅ All shape features consistent across sequences.")

---
## 2.4 — Delta feature distributions by class

⚠️ **Descriptive only** — uses `target` for colouring. Must not drive feature selection.

In [ ]:
# Identify all delta columns
# Solo le delta delle feature derivate (Step 2), non quelle radiomiche (Step 1)
delta_cols = ["delta_CE_NC_ratio", "delta_CE_vs_nadir"]

# --- Plotly: interactive boxplots ---
for col in delta_cols:
    fig = px.box(
        df, x="target", y=col, color="target",
        category_orders={"target": CLASS_ORDER},
        color_discrete_map=CLASS_PALETTE,
        points="all", title=f"{col} by RANO class",
        labels={"target": "RANO class"},
    )
    fig.update_layout(showlegend=False)
    fig.show()

# --- Seaborn: static paper figure ---
fig, axes = plt.subplots(1, len(delta_cols), figsize=(5 * len(delta_cols), 5), sharey=False)
if len(delta_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, delta_cols):
    sns.boxplot(
        data=df, x="target", y=col, hue="target",
        order=CLASS_ORDER, hue_order=CLASS_ORDER,
        palette=CLASS_PALETTE, ax=ax, width=0.5,
        legend=False, flierprops={"marker": ".", "markersize": 3},
    )
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("RANO class")
fig.suptitle("Delta feature distributions by class (descriptive only)", y=1.01)
fig.tight_layout()
fig.savefig(FIGURES / "2.4_delta_distributions.png", dpi=PAPER_FIG_DPI, bbox_inches="tight")
plt.show()

---
## 2.5 — Temporal feature distributions by class

⚠️ **Descriptive only.**

In [ ]:
# --- Plotly: interactive violin ---
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=TEMPORAL_FEATURES,
)
for i, feat in enumerate(TEMPORAL_FEATURES):
    row, col = divmod(i, 2)
    for cls in CLASS_ORDER:
        sub = df[df["target"] == cls]
        fig.add_trace(
            go.Violin(
                y=sub[feat], name=cls, legendgroup=cls,
                showlegend=(i == 0),
                line_color=CLASS_PALETTE[cls],
                box_visible=True, meanline_visible=True,
            ),
            row=row + 1, col=col + 1,
        )
fig.update_layout(title="Temporal features by RANO class (descriptive only)", height=600)
fig.show()

# --- Seaborn: static paper figure ---
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feat in zip(axes.flat, TEMPORAL_FEATURES):
    sns.violinplot(
        data=df, x="target", y=feat, order=CLASS_ORDER,
        palette=CLASS_PALETTE, ax=ax, inner="box", cut=0,
    )
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel("RANO class")
fig.suptitle("Temporal features by class (descriptive only)", y=1.01)
fig.tight_layout()
fig.savefig(FIGURES / "2.5_temporal_distributions.png", dpi=PAPER_FIG_DPI, bbox_inches="tight")
plt.show()

---
## 2.6 — Sequence length distribution

In [ ]:
seq_len = (
    df.groupby("Patient")["target"]
    .agg(["count", lambda x: x.mode()[0]])
    .rename(columns={"count": "n_timepoints", "<lambda_0>": "majority_class"})
    .reset_index()
)

print(f"Mean sequence length: {seq_len['n_timepoints'].mean():.2f}")
print(seq_len["n_timepoints"].value_counts().sort_index())

# --- Plotly: interactive histogram ---
fig = px.histogram(
    seq_len, x="n_timepoints", color="majority_class",
    color_discrete_map=CLASS_PALETTE,
    category_orders={"majority_class": CLASS_ORDER},
    nbins=15, barmode="overlay", opacity=0.7,
    title="Sequence length distribution by majority class",
    labels={"n_timepoints": "Timepoints per patient", "majority_class": "Majority class"},
)
fig.show()

# --- Seaborn: paper figure ---
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(
    data=seq_len, x="n_timepoints", hue="majority_class",
    hue_order=CLASS_ORDER, palette=CLASS_PALETTE,
    binwidth=1, multiple="dodge", ax=ax, shrink=0.8,
)
ax.set_xlabel("Timepoints per patient")
ax.set_ylabel("Count")
ax.set_title("Sequence length distribution")
fig.tight_layout()
fig.savefig(FIGURES / "2.6_sequence_length.png", dpi=PAPER_FIG_DPI)
plt.show()

---
## 2.7 — UMAP visualization

Visual sanity check. **Paper figure only — NOT model input.**  
⚠️ Descriptive only — uses `target` for colouring.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Features for UMAP: original radiomics + 9 derived (exclude boolean, target, metadata)
umap_feature_cols = [
    c for c in radiomic_cols + DERIVED_FEATURES
    if c in df.columns
    and c != "is_nadir_scan"           # boolean flag
    and df[c].dtype != object
    and df[c].notna().all()
]

X = df[umap_feature_cols].values
X_scaled = StandardScaler().fit_transform(X)  # scaling for UMAP only — not saved

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
embedding = reducer.fit_transform(X_scaled)

umap_df = pd.DataFrame({
    "UMAP1": embedding[:, 0],
    "UMAP2": embedding[:, 1],
    "class": df["target"].values,
    "Patient": df["Patient"].values,
    "scan_index": df["scan_index"].values,
})

# --- Plotly: interactive scatter ---
fig = px.scatter(
    umap_df, x="UMAP1", y="UMAP2", color="class",
    color_discrete_map=CLASS_PALETTE,
    category_orders={"class": CLASS_ORDER},
    hover_data=["Patient", "scan_index"],
    title="UMAP — radiomic + derived features (descriptive only)",
    opacity=0.7,
)
fig.show()

# --- Seaborn: paper figure ---
fig, ax = plt.subplots(figsize=(7, 6))
for cls in CLASS_ORDER:
    sub = umap_df[umap_df["class"] == cls]
    ax.scatter(sub["UMAP1"], sub["UMAP2"], label=cls,
               color=CLASS_PALETTE[cls], alpha=0.65, s=30, edgecolors="none")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title("UMAP — radiomic + derived features")
ax.legend(title="RANO class")
fig.tight_layout()
fig.savefig(FIGURES / "2.7_umap.png", dpi=PAPER_FIG_DPI)
plt.show()

---
## 2.8 — Temporal autocorrelation (t vs t+1)

Pearson correlation between consecutive timepoints for each feature.  
High corr (>0.9) → low dynamics, delta features carry most signal.  
Low corr (<0.5) → high dynamics, absolute values also informative.  
⚠️ **Descriptive only.**

In [ ]:
# _prev columns are already in the parquet from build_dataset.py
prev_cols = [c for c in df.columns if c.endswith("_prev")]
print(len(prev_cols), prev_cols[:3])
base_cols = [c.replace("_prev", "") for c in prev_cols]

# Keep only rows where _prev is not NaN (i.e. non-baseline)
df_nonbaseline = df[~df["is_baseline_scan"]].copy()

# Calcola autocorrelazione t vs t-1 per ogni feature radiomica
rc = [c for c in df.columns
      if any(f"original_{f}_" in c for f in FAMILIES)
      and not c.startswith("delta_")]

df_sorted = df.sort_values(["Patient", "time_from_diagnosis_weeks"])

autocorr_records = []
for col in rc:
    prev = df_sorted.groupby("Patient")[col].shift(1)
    # Considera solo righe non-baseline (dove shift produce un valore valido)
    mask = prev.notna()
    r = df_sorted.loc[mask, col].corr(prev[mask])
    family = next((f for f in FAMILIES if f"original_{f}_" in col), "derived")
    autocorr_records.append({"feature": col, "family": family, "autocorr": r})

autocorr_df = pd.DataFrame(autocorr_records).dropna(subset=["autocorr"])

# Family-level summary
family_summary = (
    autocorr_df.groupby("family")["autocorr"]
    .agg(["mean", "median", "std"])
    .round(3)
    .sort_values("mean", ascending=False)
)
print(family_summary.to_string())
family_summary.to_csv(FIGURES / "2.8_autocorr_family_summary.csv")

# --- Plotly: interactive distribution per family ---
fig = px.box(
    autocorr_df, x="family", y="autocorr",
    color="family",
    points="all",
    title="Temporal autocorrelation (t vs t+1) by feature family",
    labels={"autocorr": "Pearson r(t, t-1)"},
)
fig.add_hline(y=0.9, line_dash="dash", line_color="red",
              annotation_text="r=0.9 (low dynamics)", annotation_position="top left")
fig.add_hline(y=0.5, line_dash="dash", line_color="orange",
              annotation_text="r=0.5", annotation_position="bottom left")
fig.show()

# --- Seaborn: paper figure ---
family_order = family_summary.index.tolist()
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=autocorr_df, x="family", y="autocorr",
    order=family_order, ax=ax,
    flierprops={"marker": ".", "markersize": 3},
)
ax.axhline(0.9, color="red", linestyle="--", linewidth=0.8, label="r=0.9")
ax.axhline(0.5, color="orange", linestyle="--", linewidth=0.8, label="r=0.5")
ax.set_xlabel("PyRadiomics family")
ax.set_ylabel("Pearson r(t, t−1)")
ax.set_title("Temporal autocorrelation by feature family")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "2.8_temporal_autocorr.png", dpi=PAPER_FIG_DPI)
plt.show()

# Top 20 most dynamic features (lowest autocorr)
print("\nTop 20 most dynamic features (lowest autocorrelation):")
print(autocorr_df.nsmallest(20, "autocorr")[["feature", "family", "autocorr"]].to_string(index=False))

---
## Summary

All plots saved to `notebooks/figures/`.  
Findings from this notebook inform:
- **mRMR budget** in Step 3 (redundancy from 2.2)
- **Model design discussion** in paper (autocorrelation from 2.8)
- **Limitations section** in paper (sequence length from 2.6)
- **Methods section** in paper (shape consistency from 2.3)

No feature selection decisions were made here.